In [3]:
import pandas as pd
#loading data


df = pd.read_csv(r"C:\Users\natha\Downloads\project_data.csv")
#df = pd.read_csv(r"project_data.csv") # for projetc folder.


df = df.drop('c10', axis=1)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

EDA

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:

sns.set_theme(style="ticks")
sns.set_context("poster") 


g = sns.pairplot(df, hue="successful_sell", diag_kind="kde", corner=True, palette="Set2", height=2.5)


for ax in g.axes.flatten():
    if ax is not None:
        # Increase axis titles (variable names)
        ax.set_xlabel(ax.get_xlabel(), fontsize=30, fontweight='bold')
        ax.set_ylabel(ax.get_ylabel(), fontsize=30, fontweight='bold')

        ax.tick_params(axis='x', labelsize=14)
        ax.tick_params(axis='y', labelsize=14)

g.fig.suptitle("Pairwise Plot of Features by Successful Sell", y=1.02, fontsize=22, fontweight='bold')
plt.show()

Notes: N4 looks good to get a sale
low i5

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

object_cols = df.select_dtypes(include=['object']).columns
cols_to_plot = [col for col in object_cols if col != 'successful_sell']

sns.set_theme(style="whitegrid")


fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 14))
axes_flat = axes.flatten()

# Loop through columns into grid
for i, col in enumerate(cols_to_plot):
    if i < len(axes_flat):
        ax = axes_flat[i]
        sns.countplot(data=df, x=col, hue='successful_sell', palette='Set2', ax=ax)     
        ax.set_title(f'Distribution of {col}', fontsize=12)
        ax.set_xlabel(col, fontsize=20)
        ax.set_ylabel('Count', fontsize=11)      
        ax.tick_params(axis='x', rotation=90, labelsize=16)
        ax.tick_params(axis='y', labelsize=10)
        
        for p in ax.patches:
            height = p.get_height()
            if height > 0:  # Only label bars with actual counts
                ax.annotate(f'{int(height)}',
                            (p.get_x() + p.get_width() / 2., height),
                            ha='center', va='bottom',
                            fontsize=8, color='black',
                            xytext=(0, 2), textcoords='offset points')

        # Clean up individual legends
        if i > 0:
            if ax.get_legend():
                ax.get_legend().remove()
        else:
            ax.legend(title='Successful Sell', fontsize=9, title_fontsize=9)

for j in range(len(cols_to_plot), len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()


Data encoding

In [4]:
import pandas as pd

def preprocess_categorical_data(df):
    df_processed = df.copy()

    object_cols = df_processed.select_dtypes(include=['object']).columns
    #object_cols=['c3']
    # for col in object_cols:
    #     # Fill missing values with 'unknown'
    #     df_processed[col] = df_processed[col].fillna('unknown')

    cols_to_dummy = [col for col in object_cols if col != 'successful_sell']
    
    df_processed = pd.get_dummies(df_processed, columns=cols_to_dummy, drop_first=False)
    
    bool_cols = df_processed.select_dtypes(include=['bool']).columns
    df_processed[bool_cols] = df_processed[bool_cols].astype(int)
    
    return df_processed
    
df_encoded = preprocess_categorical_data(df)

Decision Stump 

In [ ]:
len(y_pred_oof)

In [ ]:
X.columns

Decision Stump

In [ ]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay
import pandas as pd

check = df_encoded.copy()
        
# Prepare actual target values as binary numeric (0/1)
target_col = 'successful_sell'
if df_encoded[target_col].dtype == 'object':
    y = df_encoded[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
else:
    y = df_encoded[target_col]

X = df_encoded.drop(columns=[target_col, 'cv_fold', 'cv_prediction'], errors='ignore')

# Set up n-fold cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# List to store global performance for every feature
all_features_summary = []
cols=['n4']
#for col in X.columns:
for col in cols:
    X_col = X[[col]]
    
    # Temporary columns to hold out-of-fold results for this feature
    oof_preds = pd.Series(index=X.index, dtype=int)
    oof_folds = pd.Series(index=X.index, dtype=int)
   
    for fold_idx, (train_index, test_index) in enumerate(kf.split(X_col)):
        fold_num = fold_idx + 1
        
        X_train, X_val = X_col.iloc[train_index], X_col.iloc[test_index]
        y_train, y_val = y.iloc[train_index], y.iloc[test_index]
        print(y_train.unique())
        print(y_train.sum())

  
        
        # Fit decision stump
        stump = DecisionTreeClassifier(max_depth=1,class_weight='balanced',random_state=42)
        stump.fit(X_train, y_train)
        tree_rules = export_text(stump, feature_names=[col])
        print("Decision Stump Rules:")
        print(tree_rules)
        
        # Predict on validation set
        y_pred = stump.predict(X_val)
        #print(sum(y_val))
        
        oof_preds.loc[X_val.index] = y_pred
        oof_folds.loc[X_val.index] = fold_num
        
        

    # Compute GLOBAL out-of-fold metrics across the entire dataset for this feature
    
    acc = accuracy_score(y, oof_preds)
    
    
    cm = confusion_matrix(y, oof_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(cmap=plt.cm.Blues)

    plt.title(f"Confusion Matrix for Stump on: {col}")
    plt.show()
    TN, FP, FN, TP = cm.ravel()
    
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    
    all_features_summary.append({
        'feature': col,
        'global_accuracy': acc,
        'true_positive_rate': tpr,
        'true_negative_rate': tnr,
        'positive_predictive_value': ppv,
        'true_positives': TP,
        'true_negatives': TN,
        'false_positives': FP,
        'false_negatives': FN
    })

# Convert summary list into a sorted global metrics DataFrame
global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='global_accuracy', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))

global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='true_positive_rate', ascending=False).reset_index(drop=True)

print(global_summary_df.head(10))

# print("--- Top 10 Features Ranked by Global Out-of-Fold Accuracy ---")
# print(global_summary_df.head(10))

DS Tree for presentations

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6), dpi=100)

# 1. Generate the standard tree plot
annotated_tree = plot_tree(
    stump, 
    feature_names=[col], 
    class_names=['No', 'Yes'], 
    filled=True, 
    rounded=True, 
    fontsize=12
)

# 2. Clean up text boxes to only keep Split Condition, Samples, and Class
for text_obj in annotated_tree:
    full_text = text_obj.get_text()
    lines = full_text.split('\n')
    
    # Filter out lines containing 'gini' or 'value'
    filtered_lines = [line for line in lines if 'gini' not in line and 'value' not in line]
    
    # Reassign the simplified text back to the tree box
    text_obj.set_text('\n'.join(filtered_lines))

plt.title(f"Simplified Decision Stump for: {col}", fontsize=14, fontweight='bold')
plt.show()

Model Selection - Best Feature

In [ ]:

ranking_metrics = [
    'global_accuracy', 
    'true_positive_rate', 
    'true_negative_rate', 
    'positive_predictive_value'
]
quantile_ranked_df = global_summary_df[ranking_metrics].rank(pct=True, ascending=True)
global_summary_df['mean_quantile_score'] = quantile_ranked_df.mean(axis=1)
quantile_sorted_summary = global_summary_df.sort_values(by='mean_quantile_score', ascending=False).reset_index(drop=True)


print("--- Top Ranked Model Runs (By Average Quantile) ---")
print(quantile_sorted_summary[['feature', 'mean_quantile_score'] + ranking_metrics].head(10))

quantile_sorted_summary.to_csv('DS_modelselection.csv') #local storage

Interactions

RandomForest

In [ ]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd

check = df_encoded.copy()
        

target_col = 'successful_sell'
if df_encoded[target_col].dtype == 'object':
    y = df_encoded[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
else:
    y = df_encoded[target_col]

X = df_encoded.drop(columns=[target_col, 'cv_fold', 'cv_prediction'], errors='ignore')

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

all_features_summary = []
#max_depth=None, min_samples_split=2, min_samples_leaf=1,max_features='sqrt'

import random
import numpy as np
runs = 10
# 1. Generate 3 random integer sequences (e.g., 5 numbers each between 1 and 50)
seq1 = [random.randint(2, 7) for _ in range(runs)]
seq2 = [random.randint(20, 900) for _ in range(runs)]
seq3 = [random.randint(2, 200) for _ in range(runs)]
seq4 = [random.randint(10, len(df_encoded.columns)) for _ in range(runs)]
seq5 = [random.randint(20, 75) for _ in range(runs)]

for i in range(runs):
    print(i)
    # 3. Package all 4 sequences into a dictionary
    # args = {
    #     #'max_depth': seq1[i],
    #     'min_samples_split': seq2[i],
    #     'min_samples_leaf': seq3[i],
    #     'max_features': seq4[i],
    #     'n_estimators':100,
    #     'random_state':1
    # }
    args = {'min_samples_split': 311, 'min_samples_leaf': 91, 'max_features': 19, 'n_estimators': 100, 'random_state': 1}
        
        # Temporary columns to hold out-of-fold results for this feature
    oof_preds = pd.Series(index=X.index, dtype=int)
    oof_folds = pd.Series(index=X.index, dtype=int)
  
    for fold_idx, (train_index, test_index) in enumerate(kf.split(X)): #loop over folds
        fold_num = fold_idx + 1
            
        X_train, X_val = X.iloc[train_index], X.iloc[test_index]
        y_train, y_val = y.iloc[train_index], y.iloc[test_index]
        #y_train = y_train.to_numpy()
        idx = y_train==1
        min_class = y_train.loc[idx==True]
        min_data = X_train.loc[idx==True,:]
        #n_times=seq5[i]
        n_times=44
        oversample_data = pd.concat([X_train] + [min_data] * n_times, ignore_index=True)
        oversample_class = pd.concat([y_train] + [min_class] * n_times, ignore_index=True)
    

            

        model = RandomForestClassifier(**args )
        model.fit(X_train, y_train)
        #model.fit(oversample_data, oversample_class)
            # tree_rules = export_text(stump, feature_names=[col])
            # print("Decision Stump Rules:")
            # print(tree_rules)
            
            # Predict on validation set
        y_pred = model.predict(X_val)
            #print(sum(y_val))
            
        oof_preds.loc[X_val.index] = y_pred
        oof_folds.loc[X_val.index] = fold_num

    acc = accuracy_score(y, oof_preds)
        
        
    cm = confusion_matrix(y, oof_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(cmap=plt.cm.Blues)
    
    plt.title(f"Confusion Matrix for Stump on: {i}")
    plt.show()
    TN, FP, FN, TP = cm.ravel()
        
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
        
    all_features_summary.append({
            'model run':i,
            'duplicates': n_times,
            'global_accuracy': acc,
            'true_positive_rate': tpr,
            'true_negative_rate': tnr,
            'positive_predictive_value': ppv,
            'true_positives': TP,
            'true_negatives': TN,
            'false_positives': FP,
            'false_negatives': FN,
            'parameters':str(args)
    })

global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='global_accuracy', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))
global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='true_positive_rate', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))

# print("--- Top 10 Features Ranked by Global Out-of-Fold Accuracy ---")
# print(global_summary_df.head(10))

Feature Importantance Plot for Presnetation

In [ ]:
# 1. Assuming `rf_model` is your trained RandomForestClassifier and `X` is your feature DataFrame
importances = model.feature_importances_
feature_names = X.columns

# 2. Create a DataFrame for easy sorting and plotting
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

# 3. Print the top 10 most important features
print("--- Top 10 Most Important Features ---")
print(feature_importance_df.head(10))

# 4. Plot the top features using Seaborn
plt.figure(figsize=(10, 6))
sns.barplot(
    x='importance', 
    y='feature', 
    data=feature_importance_df.head(10), 
    palette='Blues_r'
)
plt.title("Top 10 Random Forest Feature Importances", fontsize=16, fontweight='bold')
plt.xlabel("Mean Decrease in Impurity (Importance)", fontsize=12)
plt.ylabel("Feature Name", fontsize=12)
plt.show()

In [ ]:
Model Selection

In [ ]:
y# 1. Define the metrics to evaluate
ranking_metrics = [
    'global_accuracy', 
    'true_positive_rate', 
    'true_negative_rate', 
    'positive_predictive_value'
]

quantile_ranked_df = global_summary_df[ranking_metrics].rank(pct=True, ascending=True)


global_summary_df['mean_quantile_score'] = quantile_ranked_df.mean(axis=1)


quantile_sorted_summary = global_summary_df.sort_values(by='mean_quantile_score', ascending=False).reset_index(drop=True)


print("Top Ranked Model Runs (By Average Quantile)")
print(quantile_sorted_summary[['model run','duplicates', 'mean_quantile_score','parameters'] + ranking_metrics].head(10))
quantile_sorted_summary.to_csv('RF_modelselection.csv')

Plot of Archived Models

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


selected_metrics = [
    'global_accuracy', 
    'true_positive_rate', 
    'true_negative_rate', 
    'positive_predictive_value'
]

plot_df = global_summary_df[['model run'] + selected_metrics].copy()

g = sns.pairplot(
    plot_df, 
    hue='model run', 
    vars=selected_metrics, 
    palette='Blues', 
    diag_kind='kde'
)

g.set(xlim=(0.0, 1.0), ylim=(0.0, 1.0))

plt.suptitle("Matrix Scatter Plot (Bounded 0 to 1)", y=1.02)
plt.show()

Full Fit

In [8]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd

args = {'min_samples_split': 311, 'min_samples_leaf': 91, 'max_features': 19, 'n_estimators': 100, 'random_state': 1}

# Prepare actual target values as binary numeric (0/1)
target_col = 'successful_sell'
if df_encoded[target_col].dtype == 'object':
    y = df_encoded[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
else:
    y = df_encoded[target_col]

X = df_encoded.drop(columns=[target_col], errors='ignore')

model = RandomForestClassifier(**args )
model.fit(X, y)

import pickle
filename = 'Full_model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(model, file)




K-Means Model fitting

In [ ]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

check = df_encoded.copy()

target_col = 'successful_sell'
if df_encoded[target_col].dtype == 'object':
    y = df_encoded[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
else:
    y = df_encoded[target_col]


X = df_encoded.drop(columns=[target_col, 'cv_fold', 'cv_prediction'], errors='ignore')


n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

all_features_summary = []



#max_depth=None, min_samples_split=2, min_samples_leaf=1,max_features='sqrt'


import random
import numpy as np

runs = 1



    
# 1. Generate 3 random integer sequences (e.g., 5 numbers each between 1 and 50)
seq1 = [random.randint(2, 100) for _ in range(runs)]
mylist = ['uniform','distance']
seq2 = random.choices(mylist, k=runs)

seq5= [random.randint(2, 100) for _ in range(runs)]


for i in range(runs):
    print(i)
    # 3. Package all 4 sequences into a dictionary
    # kmeans_params = {
    #     'n_clusters': seq1[i],
    #     'random_state': 1,
    #     'n_init': 10

    # }
    kmeans_params = {
        'n_clusters': 92,
        'random_state': 1,
        'n_init': 10

    }


        
        # Temporary columns to hold out-of-fold results for this feature
    oof_preds = pd.Series(index=X.index, dtype=int)
    oof_folds = pd.Series(index=X.index, dtype=int)
    oof_clusters = pd.Series(index=X.index, dtype=int)
    
       
    for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
        fold_num = fold_idx + 1
            
        X_train, X_val = X.iloc[train_index], X.iloc[test_index]
        y_train, y_val = y.iloc[train_index], y.iloc[test_index]

        val_index = X_val.index
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val) # Or X_test if evaluating on test set
        

        
        # #y_train = y_train.to_numpy()
        # idx = (y_train == 1)
        # min_class = y_train.loc[idx]
        # min_data = X_train.loc[idx, :]
        # n_times = seq5[i]
        
        # oversample_data = pd.concat([X_train] + [min_data] * n_times, ignore_index=True)
        # oversample_class = pd.concat([y_train] + [min_class] * n_times, ignore_index=True)
            
              
            # Fit decision stump
        kmeans_model = KMeans(**kmeans_params)
        kmeans_model.fit(X_train, y_train) # Or y_train depending on your pipeline)
       
            
# Predict out of sample and retain. Need Cluster and most frequent value
        train_clusters = kmeans_model.predict(X_train)
        val_clusters = kmeans_model.predict(X_val)

        y_train_arr = y_train.to_numpy()

        cluster_mapping = {}
        for c in np.unique(train_clusters):
            mask = (train_clusters == c)
            if mask.sum() > 0:
               
                cluster_mapping[c] = int(pd.Series(y_train_arr[mask]).mode()[0])
            else:
                cluster_mapping[c] = 0

        y_pred = np.array([cluster_mapping.get(cluster, 0) for cluster in val_clusters])
        
        oof_preds.loc[val_index] = y_pred
        oof_folds.loc[val_index] = fold_num    #print(sum(y_val))
        oof_clusters.loc[val_index] = val_clusters
        
    acc = accuracy_score(y, oof_preds)
        
        
    cm = confusion_matrix(y, oof_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(cmap=plt.cm.Blues)
    
    plt.title(f"Confusion Matrix for Stump on: {i}")
    plt.show()
    TN, FP, FN, TP = cm.ravel()
        
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
        
    all_features_summary.append({
            'model run':i,
            
            'global_accuracy': acc,
            'true_positive_rate': tpr,
            'true_negative_rate': tnr,
            'positive_predictive_value': ppv,
            'true_positives': TP,
            'true_negatives': TN,
            'false_positives': FP,
            'false_negatives': FN,
            'parameters':str(kmeans_params)
    })


global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='global_accuracy', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))
global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='true_positive_rate', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))
# print("--- Top 10 Features Ranked by Global Out-of-Fold Accuracy ---")
# print(global_summary_df.head(10))

Post Hoc K-Means Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming train_clusters and y_train are available from your loop
cluster_df = pd.DataFrame({'Cluster': oof_clusters, 'Successful_Sell': y.values})

cluster_df = pd.DataFrame(df_encoded, columns=X.columns)
cluster_df['pred'] = oof_preds
cluster_df['Cluster'] = oof_clusters
cluster_df['Successful_Sell'] = y
cluster_df['good'] = cluster_df['Successful_Sell']==cluster_df.pred


target_clusters = [9, 60, 25, 14, 82, 38, 75, 19, 13, 84]

# Filter cluster_df to only include these clusters
filtered_top_clusters_df = cluster_df[cluster_df['Cluster'].isin(target_clusters)].copy()

# Plot proportion of successful sells per cluster
plt.figure(figsize=(10, 5))
sns.countplot(data=filtered_top_clusters_df, x='Cluster', hue='pred', palette='Set2')
plt.title("Target Distribution Across KMeans Clusters", fontsize=14, fontweight='bold')
plt.xlabel("Cluster ID", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title='Successful Sell', loc='upper right')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define the 6 clusters you want to plot
target_clusters = [9, 60, 25, 14, 82, 38]

# Filter cluster_df for only these clusters and ensure target/pred are strings for proper styling
multi_cluster_df = cluster_df[cluster_df['Cluster'].isin(target_clusters)].copy()
multi_cluster_df['Successful_Sell'] = multi_cluster_df['Successful_Sell'].astype(str)
multi_cluster_df['pred'] = multi_cluster_df['pred'].astype(str)

# 2. Set up a FacetGrid with 3 rows and 2 columns (col_wrap=2)
g = sns.FacetGrid(
    multi_cluster_df,
    col="Cluster",
    col_wrap=2,           # Exactly 2 columns per row -> gives a 3x2 layout for 6 clusters
    height=4,
    aspect=1.3,
    sharex=False,
    sharey=False
)

# 3. Map the scatter plot onto each grid panel
g.map_dataframe(
    sns.scatterplot,
    x='i4',
    y='i5',
    hue='pred',             # Color represents model prediction
    style='Successful_Sell', # Shape represents actual sell outcome
    palette='Set1',
    s=70,
    alpha=0.8
)

# 4. Clean up titles, labels, and legends
g.add_legend(title="Prediction / Actual", bbox_to_anchor=(1.02, 0.5), loc='center left')
g.fig.subplots_adjust(top=0.9, right=0.88)
g.fig.suptitle("i5 vs i4 Across 6 Clusters (3x2 Layout)", fontsize=16, fontweight='bold')

plt.show()

K-means table - investigations

In [ ]:
import pandas as pd

cluster_count_table = pd.crosstab(cluster_df['Cluster'], cluster_df['Successful_Sell'])
cluster_count_table.columns = ['No (0)', 'Yes (1)']
cluster_count_table.index.name = 'Cluster ID'

cluster_count_table['Total'] = cluster_count_table['No (0)'] + cluster_count_table['Yes (1)']
cluster_count_table['Conversion Ratio (%)'] = (cluster_count_table['Yes (1)'] / cluster_count_table['Total']) * 100

sorted_cluster_table = cluster_count_table.sort_values(by='Conversion Ratio (%)', ascending=False).head(10)
#sorted_cluster_table = cluster_count_table.sort_values(by='Yes (1)', ascending=False)

print("--- Cluster Target Counts & Conversion Ratio Table ---")
display(sorted_cluster_table.round(4))

In [ ]:

ranking_metrics = [
    'global_accuracy', 
    'true_positive_rate', 
    'true_negative_rate', 
    'positive_predictive_value'
]

# 2. Convert each metric into quantile/percentile ranks (pct=True gives values from 0.0 to 1.0)
# Higher values get higher ranks (ascending=True means lowest value gets lowest rank, highest gets 1.0)
quantile_ranked_df = global_summary_df[ranking_metrics].rank(pct=True, ascending=True)

# 3. Calculate the average quantile rank across all selected metrics for each model run
global_summary_df['mean_quantile_score'] = quantile_ranked_df.mean(axis=1)

# 4. Sort by the highest mean quantile score
quantile_sorted_summary = global_summary_df.sort_values(by='mean_quantile_score', ascending=False).reset_index(drop=True)

# Display the top-ranked runs based on average quantiles
print("Top Ranked Model Runs (By Average Quantile)")
print(quantile_sorted_summary[['model run', 'mean_quantile_score','parameters'] + ranking_metrics].head(10))
quantile_sorted_summary.to_csv('KM_modelselection.csv')

PCA intestigtion and Plotting

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Reduce your scaled training data to 2 dimensions using PCA
pca = PCA(n_components=3)
X_train_pca = pca.fit_transform(X_train)

# 2. Plot the K-Means clusters alongside your predicted labels
plt.figure(figsize=(10, 6))
plt.scatter(
    X_train_pca[:, 0], 
    X_train_pca[:, 1], 
    c=y_train, 
    
    alpha=0.6, 
    edgecolors='k', 
    s=30
)

plt.title(f"K-Means Clusters (n_clusters={seq1[i]}) Projected via PCA")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.colorbar(label='Cluster ID')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

XGBoost set up - not great

In [ ]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import numpy as np
import random

target_col = 'successful_sell'
if df_encoded[target_col].dtype == 'object':
    y = df_encoded[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
else:
    y = df_encoded[target_col]


X = df_encoded.drop(columns=[target_col, 'cv_fold', 'cv_prediction'], errors='ignore')

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
all_features_summary = []

runs = 10
seq1 = [random.randint(2, 12) for _ in range(runs)] # max_depth parameters

for i in range(runs):
    print(f"Running iteration {i}...")
    
    # Define XGBoost hyperparameters dictionary
    xgb_params = {
        'max_depth': seq1[i],
        'learning_rate': 0.1,
        'n_estimators': 100,
        'scale_pos_weight': 8, # Helps with class imbalance (if minority class is rare)
        'random_state': 42,
        'use_label_encoder': True,
        'eval_metric': 'logloss',
        'alpha': .1,
        'lambda': .1
    }
    
    oof_preds = pd.Series(index=X.index, dtype=int)
    oof_folds = pd.Series(index=X.index, dtype=int)
    
    for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
        fold_num = fold_idx + 1
        X_train, X_val = X.iloc[train_index], X.iloc[test_index]
        y_train, y_val = y.iloc[train_index], y.iloc[test_index]
        val_index = X_val.index
        
        # Initialize and fit XGBoost model (Tree models don't strictly require feature scaling)
        model = XGBClassifier(**xgb_params)
        model.fit(X_train, y_train)
        
        # Predict on validation data
        y_pred = model.predict(X_val)
        
        oof_preds.loc[val_index] = y_pred
        oof_folds.loc[val_index] = fold_num
        
    # Calculate global evaluation metrics
    acc = accuracy_score(y, oof_preds)
    cm = confusion_matrix(y, oof_preds, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    
    all_features_summary.append({
        'model run': i, 
        'global_accuracy': acc, 
        'true_positive_rate': tpr, 
        'true_negative_rate': tnr, 
        'positive_predictive_value': ppv, 
        'true_positives': TP, 
        'true_negatives': TN, 
        'false_positives': FP, 
        'false_negatives': FN,
        'parameters': xgb_params
    })

# Convert results summary to a DataFrame and sort by True Positive Rate
global_summary_df = pd.DataFrame(all_features_summary).sort_values(by='true_positive_rate', ascending=False).reset_index(drop=True)
print(global_summary_df.head(10))